In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from pathlib import Path

import json
import pandas as pd
import numpy as np
from collections.abc import Callable, Iterable

from jazz_graph.data.graph_builder import CreateTensors
from jazz_graph.data.fetch import fetch_recording_traits, fetch_discogs_to_recording_id
from jazz_graph.model.model import LinkPredictionModel, UnsupervisedJazzModel
from jazz_graph.recommendation.recommender import Recommender, LookupRecordings, PredictLinkRecommender
from jazz_graph.training.logging import ExperimentLogger, load_embeddings, load_model, find_most_recent_run

checkpoint_path = '../experiments/2026-03-13_20-53-49_gnn_simCLR_graph_parquet'
if checkpoint_path is None:
    checkpoint_path = str(find_most_recent_run('/workspace/experiments', 'simCLR'))

with open(Path(checkpoint_path) / 'config.json', 'r') as f:
    run_config = json.loads(f.read())
    nodes_data_path = run_config['data_config'].get('dataset')

experiment_config = {
    'gnn': checkpoint_path,
    'nodes_data': nodes_data_path,
}


In [ ]:
run_config

In [ ]:
from jazz_graph.metrics.ranking import map_at_k
from jazz_graph.recommendation.recommender import LookupRecordings

class RecommenderExperiment:
    def __init__(self, recommender: Recommender, recording_traits: pd.DataFrame):
        self.recommender = recommender
        self.recording_traits = recording_traits

    def _extract_recording_ids(self, artist, album) -> tuple:
        album_recordings = self.recording_traits.query(f"artist == '{artist}'").query(f"album == '{album}'").index.to_numpy()
        assert len(album_recordings) > 1, f"Unable to identify mulitple recordings with {artist}, {album}"
        n_inputs = len(album_recordings) // 2
        input_ids = album_recordings[:n_inputs]
        expected_recs = album_recordings[n_inputs:]
        return input_ids, expected_recs

    def b_side_precision(self, artist: str, album: str):
        """Do a b-side experiment for the input artist's album."""
        input_ids, expected_recs = self._extract_recording_ids(artist, album)
        k = len(expected_recs) + 20
        recs, _, mask = self.recommender.get_recommendations(input_ids.tolist())
        recs = recs[~mask]
        map_k = map_at_k(recs, expected_recs, k)
        return {
            'artist': artist,
            'album': album,
            'n_inputs': len(input_ids),
            'n_expected_recs': len(expected_recs),
            'k': k,
            f'MAP_at_k': float(map_k),
            'recommended_ids': recs.tolist()[:k]}

    def b_side_experiment(self, experiment_config: dict, album_experiments: list[tuple[str, str]]):
        """Perform a b-side experiment for each artist-album pair.

        The second side of a record, also known as the 'B-side' should probably be
        recommended given only the A-side as inputs. This experiment simulates that
        as a basic sanity check for the recommender.
        """
        experiment_log = ExperimentLogger(root='../experiments', run_name='recommendations', config=experiment_config)
        results = []
        for artist, album in album_experiments:
            result = self.b_side_precision(artist, album)
            experiment_log.log_metrics(None, result, "b_side_experiment")
            results.append(result)
        return results


def lookup_from_dataset(data: HeteroData) -> LookupRecordings:
    rec_ids = data['performance'].x[:, 1].numpy()
    ids = np.arange(rec_ids.size)
    df = pd.DataFrame({'recording_ids': rec_ids, 'ids': ids}).set_index('recording_ids')
    return LookupRecordings(df)


In [ ]:
class UnsupervisedModelAdapter(torch.nn.Module):
    def __init__(self, model: UnsupervisedJazzModel):
        super().__init__()
        self.model = model

    def __call__(self, x_dict, edge_index_dict, batch):
        return self.model(batch)
        return self.model.encode(batch)

In [ ]:
def iter_dirs(root: str):
    import os
    for base_dir in os.listdir(root):
        print(base_dir)
        break

iter_dirs('/workspace/experiments')

In [ ]:
## Inductive Graph Recommender
from sklearn import model_selection
from jazz_graph.data.graph_builder import make_jazz_data
from jazz_graph.model.model import JazzModel
from jazz_graph.recommendation.recommender import InferenceRecommender


graph_data: HeteroData = make_jazz_data(CreateTensors(nodes_data_path))
model_state = load_model(checkpoint_path)
model_state = model_state.get('model_state_dict', model_state)
# model_state = model_state.get('model_state_dict', model_state)
model = UnsupervisedJazzModel.from_config(run_config)
model.load_state_dict(model_state)
model = UnsupervisedModelAdapter(model)
lookup = lookup_from_dataset(graph_data)
recommender = InferenceRecommender(model, graph_data, lookup)
recording_traits = fetch_recording_traits(use_proto=nodes_data_path.endswith('proto')).set_index('recording_id')
experimenter = RecommenderExperiment(recommender, recording_traits)

In [ ]:

album_experiments = [
    ('Miles Davis', 'Kind of Blue'),
    ('Miles Davis', 'Sketches of Spain'),
    ('Art Blakey & The Jazz Messengers', 'Mosaic'),
    ('Charles Mingus', "Mingus Ah Um"),  # lots of songs, should have some.
    ('The Dave Brubeck Quartet', "Time Out"),
    ('Ornette Coleman', 'The Shape of Jazz to Come')  # very unusual music--should probably be easy.
]

In [ ]:
run_experiment = album_experiments[-2]
print(run_experiment)
result = experimenter.b_side_precision(*run_experiment)
result
recording_traits.loc[result['recommended_ids']]

In [ ]:
results = experimenter.b_side_experiment(experiment_config, album_experiments)
pd.DataFrame.from_records(results).drop(columns=['recommended_ids'])

## Spotify Data

In [ ]:
import os
import json
from jazz_graph.recommendation.playlist import SpotifyListens
from jazz_graph.clean.data_normalization import normalize_title
import random


In [ ]:
spotify = 'local_data'
spotify_sample = '../local_data/my_spotify_data/spotify_extended_streaming_history/Streaming_History_Audio_2025-2026_5.json'
with open(spotify_sample, 'r') as f:
    spotify_data = json.loads(f.read())

def spotify_details(record: dict):
    details = [
        'ts', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name',
        'reason_start', 'reason_end', 'shuffle', 'skipped'
    ]
    return {key: record.get(key) for key in details}

spotify_data[-100:-90]
spotify_data[-10:]
ten_recent = [spotify_details(record) for record in spotify_data[-20:-10]]

In [ ]:

seen = set()
i = 0
for rec in spotify_data:
    title = rec['master_metadata_track_name']
    norm_title = normalize_title(title)
    if norm_title in seen:
        continue
    seen.add(norm_title)
    if 'digital' in title.lower():
        i += 1
        print(norm_title)
        print(title)
    # print(norm_title)
print(i)

In [ ]:
class RandomAlbumSplit:
    def __init__(self, frac: float = .5, seed: int|None = None):
        self.frac = frac
        self.split_a = set()
        self.split_b = set()
        self.recordings_a = []
        self.recordings_b = []
        if seed is not None:
            random.seed(seed)

    def make_splits(self, data: Iterable):
        for record in data:
            self.add_to_splits(record)

    def add_to_splits(self, record):
        album = record.release_group_id
        recording_id = record.Index
        if album in self.split_a:
            self.recordings_a.append(recording_id)
        elif album in self.split_b:
            self.recordings_b.append(recording_id)
        else:
            if random.random() > self.frac:
                self.recordings_a.append(recording_id)
                self.split_a.add(album)
            else:
                self.recordings_b.append(recording_id)
                self.split_b.add(album)


In [ ]:
from collections import namedtuple

Record = namedtuple('Record', ['Index', 'release_group_id'])

class TestRandomAlbumSplit:

    def test_add_to_splits(self):
        album_split = RandomAlbumSplit(seed=2)
        record = {'recording_id': 1, 'album_id': 10}
        record = Record(1, 10)
        album_split.add_to_splits(record)
        assert 10 in album_split.split_a or 10 in album_split.split_b
        if 10 in album_split.split_a:
            assert 1 in album_split.recordings_a
        else:
            assert 1 in album_split.recordings_b

        record = Record(2, 10)
        album_split.add_to_splits(record)
        if 10 in album_split.split_a:
            assert 2 in album_split.recordings_a
        else:
            assert 2 in album_split.recordings_b

    def test_make_splits(self):
        album_split = RandomAlbumSplit(seed=2)
        data = [Record(x * 1000, x % 100) for x in range(100)]
        album_split.make_splits(data)
        for x in album_split.recordings_a:
            x_root = x // 1000
            assert x_root % 100 in album_split.split_a
        for x in album_split.recordings_b:
            x_root = x // 1000
            assert x_root % 100 in album_split.split_b

        assert album_split.split_a, "There should be some albums in a."
        assert album_split.split_b, "There should be some albums in b."
        assert not album_split.split_a.intersection(album_split.split_b), "They should not overlap."




TestRandomAlbumSplit().test_add_to_splits()
TestRandomAlbumSplit().test_make_splits()


In [ ]:
class SpotifyExperiement:
    def __init__(self, recording_traits: pd.DataFrame, listening_history: list[dict]):
        self.listening_history = listening_history
        self.spotify = SpotifyListens(recording_traits)
        self.album_split = RandomAlbumSplit()
        self.album_split.make_splits(self.spotify.get_listen_data(listening_history))

    def run_experiment(self, recommender: Recommender, experiment_config):
        recommendations, scores, mask = recommender.get_recommendations(self.in_samples)
        metrics = self.experiment_metrics(recommendations, mask)
        self.log_experiment(experiment_config, metrics)

    @property
    def in_samples(self) -> np.ndarray:
        return np.array(self.album_split.recordings_a)

    def _make_logger(self, experiment_config):
        self._experiment_log = ExperimentLogger(root='../experiments', run_name='spotify_recommendations', config=experiment_config)

    def log_experiment(self, experiment_config, metrics):
        if not hasattr(self, '_experiment_log'):
            self._make_logger(experiment_config)
        else:
            config = self._experiment_log.config()
            config.pop('commit')    # pyright: ignore [reportOptionalMemberAccess]  <- None should never happen here, so fail if it does, but not type error.
            if experiment_config != config:
                self._make_logger(experiment_config)
        experiment_log = self._experiment_log
        experiment_log.log_metrics(None, metrics, "spotify_recommendations")


    @property
    def out_samples(self) -> np.ndarray:
        return np.array(self.album_split.recordings_b)

    def experiment_metrics(self, recommendations, mask):
        # We are mainly interested in recall of out-of-sample splits from the album spliter.
        # Recall of albums from the in-sample cases are less interesting, since no matter
        # how the model scores those, we can always filter--the familiar candiates are availbale
        # at inference time.
        in_samples = self.in_samples
        out_samples = self.out_samples
        novel_recommendataions = recommendations[~mask]
        familiar_recommendations = recommendations[mask]

        n = max(20, out_samples.size * 2)
        positives_novel = novel_recommendataions[:n]
        postives_familiar = familiar_recommendations[:n]

        novel_tp = np.intersect1d(novel_recommendataions, out_samples)
        familiar_tp = np.intersect1d(familiar_recommendations, in_samples)
        novel_fn = np.setdiff1d(out_samples, novel_tp)
        familiar_fn = np.setdiff1d(in_samples, familiar_tp)

        recall_novel = novel_tp.size / (novel_tp.size + novel_fn.size)
        recall_familiar = familiar_tp.size / (familiar_tp.size + familiar_fn.size)

        metrics = {
            'recall_novel': float(recall_novel),
            'recall_known': float(recall_familiar),
            'recall_tp': float(novel_tp.size),
            'known_tp': float(familiar_tp.size),
            'samples': {
                f'top_{n}_recommendations (excl. known)': positives_novel.tolist(),
                'true_positive_in_novel_recommendation': novel_tp.tolist(),
                'false_negative_in_novel_sample': novel_fn.tolist(),
            }

        }
        return metrics


In [ ]:
class RandomWalkRecommender:
    """Recommend performances by a two-hop random walk of the graph.

    The walk hops via any relation edge to an adjacent node then hops back to
    any node of the same type as the origin node.

    Used as a baseline model for making recommendations.
    """
    def __init__(self, data: HeteroData, lookup: LookupRecordings):
        self.data = data
        self.node_type = 'performance'
        self.lookup_recordings = lookup

    def get_recommendations(self, listens):
        """Get recommended recording_ids, their scores and a mask of input ids."""
        node_ids = self.lookup_recordings.lookup_node_index(listens)
        rec_node_ids = self.heterogeneous_two_hop_random_walk(self.data, self.node_type, torch.from_numpy(node_ids))
        recording_ids = self.lookup_recordings.lookup_recording_ids(rec_node_ids.numpy())
        mask = np.isin(recording_ids, listens)
        return recording_ids, np.zeros_like(recording_ids), mask

    @staticmethod
    def heterogeneous_two_hop_random_walk(
        data: HeteroData,
        node_type: str,
        node_indices: torch.Tensor,
        num_walks: int = 10,
        return_intermediate: bool = False,
    ) -> torch.Tensor:
        # Claude.ai provided this code.
        """
        Perform two-hop random walks on a HeteroData graph, starting and ending
        at nodes of the same type (origin -> intermediate -> origin-type node).

        Each walk follows one outgoing (or incoming) edge to an intermediate node
        of a different type, then follows one edge back to a node of the original
        type. The destination may be the origin node itself.

        Args:
            data:               A PyTorch Geometric HeteroData object.
            node_type:          The starting (and ending) node type, e.g. 'artist'.
            node_indices:       1-D tensor of node indices to start walks from.
            num_walks:          Number of walks to attempt per source node.
            return_intermediate: If True, also return the intermediate node tensor.

        Returns:
            walks: LongTensor of shape (N * num_walks, 2) where columns are
                [source_node_index, destination_node_index], both in the
                coordinate space of `node_type`. Invalid/dead-end walks are
                represented as -1.
            (optional) intermediates: LongTensor of shape (N * num_walks,) with
                the intermediate node index visited, -1 for dead ends.
        """
        node_indices = node_indices.long()
        N = node_indices.size(0)
        total = N * num_walks

        # Expand source indices so each node appears num_walks times.
        sources = node_indices.repeat_interleave(num_walks)  # (total,)

        # Collect all edge relations that touch `node_type`.
        # We separate them into:
        #   forward: node_type -[rel]-> other_type
        #   backward: other_type -[rel]-> node_type  (we traverse in reverse)
        forward_edges = []  # (src_indices, dst_indices, other_type)
        backward_edges = []  # (other_indices, dst_indices, other_type) stored as
        #                      other->node_type adjacency for the return hop

        for (src_type, rel, dst_type), edge_index in data.edge_index_dict.items():
            if src_type == node_type and dst_type != node_type:
                forward_edges.append((edge_index, dst_type, "forward"))
            if dst_type == node_type and src_type != node_type:
                # Reverse traversal: we can walk *backward* along this edge
                forward_edges.append((edge_index.flip(0), src_type, "backward"))

        if not forward_edges:
            raise ValueError(f"No edges found leaving node type '{node_type}'.")

        # Build adjacency dicts for fast neighbour lookup.
        # adj_out[other_type] : dict[int -> Tensor]  node_type node -> neighbours
        # adj_back[other_type]: dict[int -> Tensor]  other_type node -> node_type nodes
        adj_out: dict[str, dict] = {}
        adj_back: dict[str, dict] = {}

        for (src_type, rel, dst_type), edge_index in data.edge_index_dict.items():
            srcs, dsts = edge_index[0], edge_index[1]

            if src_type == node_type and dst_type != node_type:
                if dst_type not in adj_out:
                    adj_out[dst_type] = {}
                for s, d in zip(srcs.tolist(), dsts.tolist()):
                    adj_out[dst_type].setdefault(s, []).append(d)
                # Return hop: from dst_type back to node_type
                if dst_type not in adj_back:
                    adj_back[dst_type] = {}
                for s, d in zip(dsts.tolist(), srcs.tolist()):
                    adj_back[dst_type].setdefault(s, []).append(d)

            if dst_type == node_type and src_type != node_type:
                # Can also enter via reverse: node_type <- other_type
                if src_type not in adj_out:
                    adj_out[src_type] = {}
                for s, d in zip(dsts.tolist(), srcs.tolist()):
                    adj_out[src_type].setdefault(s, []).append(d)
                if src_type not in adj_back:
                    adj_back[src_type] = {}
                for s, d in zip(srcs.tolist(), dsts.tolist()):
                    adj_back[src_type].setdefault(s, []).append(d)

        dest_nodes = torch.full((total,), -1, dtype=torch.long)
        inter_nodes = torch.full((total,), -1, dtype=torch.long)

        other_types = list(set(adj_out.keys()) & set(adj_back.keys()))
        if not other_types:
            raise ValueError(
                f"No round-trip paths found for node type '{node_type}'."
            )

        for i, src in enumerate(sources.tolist()):
            # --- Hop 1: pick a random intermediate type, then a random neighbour ---
            # Gather all reachable intermediate nodes across all relation types.
            candidates: list[tuple[str, int]] = []
            for ot in other_types:
                if src in adj_out[ot]:
                    for nb in adj_out[ot][src]:
                        candidates.append((ot, nb))

            if not candidates:
                continue  # dead end — stays -1

            rand_idx = torch.randint(len(candidates), (1,)).item()
            inter_type, inter_node = candidates[rand_idx]
            inter_nodes[i] = inter_node

            # --- Hop 2: from intermediate node back to node_type ---
            return_candidates = adj_back[inter_type].get(inter_node, [])
            if not return_candidates:
                continue  # dead end on return

            rand_idx2 = torch.randint(len(return_candidates), (1,)).item()
            dest_nodes[i] = return_candidates[rand_idx2]

        walks = torch.stack([sources, dest_nodes], dim=1)  # (total, 2)

        if return_intermediate:
            return walks, inter_nodes
        return walks


def filter_valid_walks(walks: torch.Tensor) -> torch.Tensor:
    """Remove walks that hit a dead end (destination == -1)."""
    return walks[walks[:, 1] != -1]
# ```

# **How it works for your schema:**

# The walk follows a two-hop pattern. Starting from, say, an `artist` node, hop 1 traverses *any* edge that connects `artist` to another type — so it can land on either a `performance` (via `performs`) or a `song` (via `composes`). Hop 2 then traverses back along any edge that returns to `artist`. The schema's bidirectional relations make this natural:
# ```
# artist → (performs) → performance → (performs, reversed) → artist
# artist → (composes) → song        → (composes, reversed) → artist
# artist → (performs) → performance → (performing, reversed) → ...
#   [dead end — performance→song has no return to artist]

In [ ]:
experiment = SpotifyExperiement(recording_traits, spotify_data)

In [ ]:
experiment.run_experiment(recommender, experiment_config)


In [ ]:
novel_recs = recording_traits.loc[recommendations[~mask]][:500]
all_recs = recording_traits.loc[recommendations][:500].assign(rank=np.arange(1, 501))
novel_recs = novel_recs.merge(all_recs['rank'], left_index=True, right_index=True)
repeat_idx = all_recs.index.difference(novel_recs.index)
repeat_recs = all_recs.loc[repeat_idx]
repeat_recs.head(30)

Sanity Check:
- Scores: do obviously similar performances (e.g. same album) cluster? 
- Provide half of the songs from an album. Are the other half highly rated? If not, indicates bug or surprising learning by the model.
Pick an album where the compositions are mostly novel ones. See Sketches of Spain above; that seemed a little off to me.

Bad candidates indicate that cosine similarity between the mean performance vectors of the listener combined with the performance embeddings don't produce a reliable score. 
Bad embeddings: If performance embeddings don't cluster much, we would expect this. Could also be that what he GNN is learning isn't very relevant. (we would still expect songs from side B of an album to be high given songs from side A.) Indicates issue with GNN.
Bad aggregation: It could be that there are issues with taking the mean of the listener performance. Indicates that we need better user model.
Bad similarity score: It could be that dot product rather than similarity is the way to go. Switch similarity metric, possibly as alpha(CS) + (1  - alpha)(DP) Question: what does the embedding norm mean? That's not an easy question to answer...
Bug: this is the most annoying possibility: what if there's a problem in your set up and your not aligning embeddings, etc. correctly?
you would expect random seeming recommendation in that case. <- This is unlikely. The album precision experiments show high relevance for the input albums but much lower relevance to the B-sides.
